# Análisis exploratorio de vuelos (SDQ)

Análisis exploratorio en Python del proyecto. No incluye datos ni salidas: para ejecutarlo se necesita una base propia.

## 1. Conexión a la base

Se conecta con el usuario de solo lectura mediante variables de entorno y cuenta los vuelos por dirección.

In [ ]:
import os

from sqlalchemy import create_engine, URL
import pandas as pd

engine = create_engine(URL.create(
    "mysql+pymysql",
    username=os.environ["VUELOS_RO_USER"], password=os.environ["VUELOS_RO_PASSWORD"],
    host=os.environ["VUELOS_DB_HOST"], database=os.environ["VUELOS_DB_NAME"]))

pd.read_sql("SELECT direction, COUNT(*) AS vuelos FROM flight_movements GROUP BY direction", engine)

## 2. Preparación de los datos

Lee los vuelos y calcula la hora local de SDQ y el retraso en minutos.

In [ ]:
q = """
SELECT direction, flight_number, airline_name, other_airport_name,
       status, sched_utc, revised_utc, runway_utc, last_seen_utc
FROM flight_movements
"""
df = pd.read_sql(q, engine, parse_dates=["sched_utc", "revised_utc", "runway_utc", "last_seen_utc"])

# Hora local de SDQ (UTC-4, sin horario de verano)
df["sched_local"] = df["sched_utc"] - pd.Timedelta(hours=4)
df["hora_local"] = df["sched_local"].dt.hour

# Retraso en minutos: pista si ya ocurrió, si no la revisada (misma regla que daily_summary.R)
real = df["runway_utc"].where(df["runway_utc"] <= df["last_seen_utc"], df["revised_utc"])
df["retraso_min"] = (real - df["sched_utc"]).dt.total_seconds() / 60

df.head()

## 3. Retraso por aerolínea

Excluye cancelados, desviados y retrasos de más de 720 minutos, y calcula el retraso promedio y el % de vuelos con más de 15 minutos.

In [ ]:
import matplotlib.pyplot as plt

validos = df[~df["status"].isin(["Canceled", "CanceledUncertain", "Diverted"])]
con_dato = validos[validos["retraso_min"].notna() & (validos["retraso_min"] <= 720)]

por_aerolinea = (
    con_dato.groupby("airline_name")
    .agg(vuelos_con_dato=("retraso_min", "size"),
         retraso_prom=("retraso_min", "mean"),
         pct_mas_15=("retraso_min", lambda s: (s > 15).mean() * 100))
    .round(1)
    .sort_values("retraso_prom", ascending=False)
)
display(por_aerolinea)

por_aerolinea["retraso_prom"].plot(kind="barh", color="#1D9E75", figsize=(8, 4))
plt.xlabel("Retraso promedio (min)")
plt.title("Retraso promedio por aerolínea — SDQ (Datos: AeroDataBox)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()